# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIRˆ2 Exploration with `mlcroissant`
This notebook provides a guided template for loading, exploring, and processing the [FAIRˆ2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library. The dataset includes clinicopathological details (demographics, comorbidities, tumor characteristics, molecular markers, and more) for 77 cancer survivors with a second primary colorectal cancer (CRC). All dataset entities are referenced by their Croissant `@id` identifiers throughout.

### Dataset Source
The dataset is described by its FAIR Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed in your environment
!pip install mlcroissant

## 1. Data Loading
Load the Croissant metadata and tabular records with `mlcroissant`. We'll also review primary dataset information and context.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# The Croissant schema URL for FAIR^2
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata structure
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is accessed as an object, not subscripted

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier if hasattr(metadata, 'identifier') else 'N/A'}")
print(f"Publication Date: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")
print(f"Number of Record Sets: {len(metadata.record_sets)}")

## 2. Data Overview
We enumerate available Record Sets, list their Croissant `@id`'s, and preview their field structures. Each Record Set and each field is referenced by its Croissant `@id`. (If documentation is unclear or sets are not named, we'll load the only/main Record Set.)

In [ ]:
# List all Record Sets and their Fields, referencing by @id
record_sets = metadata.record_sets
print(f"Available Record Sets (@id):")
for rs in record_sets:
    print(f"  - {rs.id} (name: {getattr(rs, 'name', 'N/A')})")
    print("    Fields:")
    for field in rs.fields:
        print(f"      * {field.id} (name: {getattr(field, 'name', 'N/A')}, dataType: {getattr(field, 'dataType', 'N/A')})")

## 3. Data Extraction
We'll load data for each available record set by referencing the record set `@id`. All columns/fields are preserved with their source entity `@id`s for traceability.

*Tip:* Replace `<your_record_set_id>` with a specific `@id` as needed for narrower exploration.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in record_sets]

dfs = {}
# Load each record set into a dataframe referenced by its @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dfs[record_set_id] = df
    print(f"Loaded '{record_set_id}' with shape {df.shape}")

# Select the primary record set for illustration (most datasets only have one)
primary_record_set_id = record_set_ids[0]
print(f"\nColumns in Record Set '{primary_record_set_id}':\n{dfs[primary_record_set_id].columns.tolist()}")
dfs[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field to demonstrate basic filtering, normalization, and grouping. All fields are referenced by Croissant `@id`.

**Choose field(s) to analyze:**
- The field `cr:age_second_crc` (or similar) refers to patients' age at second CRC diagnosis (if present).
- The field `cr:interval_between_cancers` (if defined) refers to the interval (years or months) between first and second cancer.
- For demonstration, we'll auto-select the first numeric field present.

In [ ]:
# Find a numeric field (float or integer) in the primary record set
primary_fields = [
    (f.id, f.name, f.dataType)
    for f in [rs for rs in record_sets if rs.id == primary_record_set_id][0].fields
]
# Typically, numeric fields have dataType 'schema:Integer' or 'schema:Float'
numeric_field_candidates = [fid for fid, fname, dtype in primary_fields if str(dtype).lower() in ('schema:integer', 'schema:float', 'integer', 'float', 'number')]

if not numeric_field_candidates:
    print("No numeric field found for analysis.")
else:
    # Use the first numeric field found
    numeric_field_id = numeric_field_candidates[0]
    print(f"Using numeric field: {numeric_field_id}")


    # Example filter: value > 60 (for age, e.g.)
    threshold = 60
    df = dfs[primary_record_set_id]
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize that field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a categorical/grouping field (for example, 'cr:sex' or 'cr:msi_status')
        group_field_candidates = [fid for fid, fname, dtype in primary_fields if str(dtype).lower() not in ('schema:integer', 'schema:float', 'integer', 'float', 'number')]
        group_field_id = group_field_candidates[0] if group_field_candidates else None

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"\nGrouped data by {group_field_id} (showing mean {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print(f"Field {numeric_field_id} not found in dataframe columns.")

## 5. Visualization
Let's visualize the distribution of our selected numeric field, using the Croissant `@id` as column reference.

*For more advanced visualizations, you can group or facet by a categorical `@id` field such as `cr:sex` or `cr:msi_status` if present in your dataset.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram and boxplot for the selected numeric field
if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")

    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()

    # If a group_field was found, show its grouped barplot
    if 'group_field_id' in locals() and group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(7,4))
        sns.barplot(
            x=group_field_id, y=numeric_field_id,
            data=df,
            ci='sd'
        )
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and process a FAIR dataset published in Croissant format, using the `mlcroissant` Python library. All dataset entities are referenced by their Croissant `@id` to ensure future-proof, schema-consistent analyses.

- We loaded the metadata and identified the available record sets and fields by `@id`.
- We extracted primary records and performed basic EDA and simple visualizations.
- Best practices encourage referencing any field, column, or record set by `@id`, not by display name, to ensure unambiguous and version-stable analyses.

**Next steps**: You can customize further, building ML pipelines, or deeper statistical analyses. For further information, see [`mlcroissant` documentation](https://mlcommons.github.io/croissant/python/).
